# GenAI-Kickstart
## Lesson: RAG Optimization

This lesson is intended to show you how to optimize a RAG system.

During this lesson you will learn how to ...

- swap the embedding model
- use **query rewriting** to reshape the user's question into a more retrieval-friendly query
- use **HyDE** to generate hypothetical documents and embed those instead of the raw query
- (optional) use a cross-encoder **reranker** to reorder retrieved chunks by task-specific relevance


### Set up the environment
Import the necessary libraries, set constants, and define helper functions.

In [1]:
import sys
sys.path.append('../../../')

from dotenv import load_dotenv, find_dotenv
from chromadb import EphemeralClient
from lib.rag import do_embed, do_ingestion
from lib.completion import generate_completion
from lib.constants import PROCESSED_DATA_PATH, EVALUATION_DATA_PATH
from lib.evaluation import (
    FetchedChunk, do_top_k_fetching, load_testdata_from_csv, run_evaluation,
)
from lib.validation import validate_environment_for_model
import os

load_dotenv(find_dotenv())

True

In [2]:
# Set values for model, model parameters and prompt

# Models
GENERATION_PROVIDER = os.getenv("DEFAULT_GENERATION_PROVIDER")
GENERATION_MODEL = os.getenv("DEFAULT_GENERATION_MODEL")
EMBEDDING_PROVIDER = os.getenv("DEFAULT_EMBEDDING_PROVIDER")
EMBEDDING_MODEL = os.getenv("DEFAULT_EMBEDDING_MODEL")

# Generation parameters
DEFAULT_CONFIG_TEMPERATURE = 0.9
DEFAULT_CONFIG_MAX_OUTPUT_TOKENS = 200
DEFAULT_SYSTEM_PROMPT = "Your are a friendly assistant"

# Retrieval parameters
DEFAULT_K = 10
DEFAULT_CHUNK_SIZE = 1000
DEFAULT_CHUNK_OVERLAP = 100
DEFAULT_BATCH_SIZE = 100

In [3]:
# Set the full model names
generation_model = f"{GENERATION_PROVIDER}/{GENERATION_MODEL}"
embedding_model = f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}"

# Check if all environment variables are set
validate_environment_for_model(model=generation_model)
validate_environment_for_model(model=embedding_model)

In [4]:
# Helper function to get the page number of a specific string in the book text
def get_page_number(text: str, book_text: str, characters_per_page: int = 1800) -> int:
    """
    Finds the page number of a specific string in the book text.

    Parameters
    ----------
    text : str
        The string to search for.
    book_text : str
        The complete book text.
    characters_per_page : int, optional
        The approximate number of characters per page, by default 1800.

    Returns
    -------
    int
        The page number where the string is found, or -1 if not found.
    """
    start_index = book_text.find(text)
    if start_index == -1:
        return -1
    page_number = (start_index // characters_per_page) + 1
    return page_number

In [5]:
# This will be the chromadb collection we use as a knowledge base.
chromadb_client = EphemeralClient()

### Evaluate the basic RAG to define the baseline

In [6]:
# We will optimize only the retrieval of the RAG system, therefore we do not augment or generate a response
def do_rag(user_input: str, top_k: int, verbose: bool = False) -> list[FetchedChunk]:
  """ Runs the retrieval pipeline with a given user input and returns the context

  Parameters
  ----------
  user_input : str
      The user input to be used for the RAG pipeline
  top_k : int
      The top_k closest chunks will be retrieved
  Returns
  -------
  context: [FetchedChunk]
      The fetched chunks
  """

  # Embed the hypothetical book summaries
  user_input_embedding = do_embed(user_input, embedding_model)

  # "R" like "Retrieval": Get the top-k semantically closest chunks to the hypothetical summary
  return do_top_k_fetching(user_input_embedding=user_input_embedding, top_k=top_k, chromadb_client=chromadb_client)

In [7]:
# Prepare the knowledgebase
KNOWLEDGEBASE_CONTENT = ["study_in_scarlett.txt", "adventures_of_sherlock_holmes.txt"]
do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=PROCESSED_DATA_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
    chunk_size=DEFAULT_CHUNK_SIZE,
    chunk_overlap=DEFAULT_CHUNK_OVERLAP
)

In [11]:
# Load the testdata
testdata_items = load_testdata_from_csv(path=EVALUATION_DATA_PATH/ "evaluation_dataset.csv")

In [13]:
# Evaluate the retrieval without optimizations
evaluation_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=testdata_items,
    rag_function=do_rag
)

print("Baseline evaluation results:")
print(f"Mean Hit Rate: {evaluation_result.mean_hit_rate:.2f}")
print(f"Mean Reciprocal Rank: {evaluation_result.mean_reciprocal_rank:.2f}")
print(f"Mean Precision: {evaluation_result.mean_precision:.2f}")

Baseline evaluation results:
Mean Hit Rate: 0.60
Mean Reciprocal Rank: 0.52
Mean Precision: 0.06


### Exercise 01: Switch the embedding model
Up to this point, we used only one mbedding model. We might get better results using a different model. Thanks to litellm, switching the embedding model is as simple as changing the provider and model name.  
Your task is to try a different embedding model via Ollama and evaluate the retrieval.

You can try the following models:
- `qwen3-embedding:0.6b`
- `nomic-embed-text`
- `embeddinggemma`


In [14]:
# TODO: Switch the ollama embedding model
EMBEDDING_PROVIDER = os.getenv("DEFAULT_EMBEDDING_PROVIDER")
EMBEDDING_MODEL = "nomic-embed-text"
embedding_model = f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}"

validate_environment_for_model(model=embedding_model)

In [15]:
# Ingest the books using the new embedding model
do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=PROCESSED_DATA_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
)

In [16]:
# Evaluate the retrieval without optimizations
evaluation_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=testdata_items,
    rag_function=do_rag
)

print(f"Mean Hit Rate: {evaluation_result.mean_hit_rate:.2f}")
print(f"Mean Reciprocal Rank: {evaluation_result.mean_reciprocal_rank:.2f}")
print(f"Mean Precision: {evaluation_result.mean_precision:.2f}")

Mean Hit Rate: 0.70
Mean Reciprocal Rank: 0.53
Mean Precision: 0.07


### Reset embeddings to the configured model
Execute this as soon as you are ready to move on to the next exercise.

In [17]:
# Reset to the best embedding model and re-ingest
EMBEDDING_MODEL = os.getenv("DEFAULT_EMBEDDING_MODEL")
embedding_model = f"{EMBEDDING_PROVIDER}/{EMBEDDING_MODEL}"

do_ingestion(
    file_names=KNOWLEDGEBASE_CONTENT,
    data_path=PROCESSED_DATA_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
)

### Exercise 02: Query Rewriting
A user's natural-language question often doesn't phrase concepts the way the source corpus does. An LLM can rewrite the question into a more retrieval-friendly form — resolving pronouns, expanding acronyms, dropping conversational filler, and surfacing entities and keywords.

Think of this as **HyDE-lite**: same idea of putting an LLM in front of retrieval, but the output stays at the *query* level instead of fabricating a hypothetical answer document. It's cheaper (one short generation), and it avoids inventing facts that could anchor retrieval to the wrong neighborhood.

**Your task:** implement `rewrite_query` so that it turns a user question into a clean search query, then evaluate `do_query_rewrite_rag` against the same testdata.

In [18]:
# TODO: Implement a function which rewrites a user question into a more retrieval-friendly query
def rewrite_query(user_input: str) -> str:
    """Rewrites a user question into a search-friendly query

    Parameters
    ----------
    user_input : str
        The user input to be rewritten
    Returns
    -------
    response: str
        The rewritten query
    """

    # TODO: Use your prompt engineering skills to create a prompt that turns
    # a user's natural-language question into a clean search query.
    #
    # The rewritten query should:
    #   - Preserve the user's intent and any named entities
    #     (Holmes, Watson, Lucy, RACHE, John Ferrier, etc.)
    #   - Resolve pronouns and remove conversational filler
    #     ("could you tell me", "I want to know", etc.)
    #   - Be phrased as a declarative search query, not a question
    #   - Stay one short sentence
    #   - Be returned as plain text only — no quotes, no preamble
    #
    # As an example, you can use the following user question and the
    # resulting rewritten query:
    #
    # USER QUESTION:
    # "Could you tell me what the word was that Holmes saw written on the wall in blood?"
    #
    # REWRITTEN QUERY:
    # The word written in blood that Sherlock Holmes found on the wall at the crime scene

    rewrite_prompt = f"""
      You are a search-query rewriter for a Sherlock Holmes corpus.
      Given a user question, rewrite it into a single, concise search query
      that is more likely to match the phrasing of the source text.

      Rules:
        - Preserve the user's intent and all named entities exactly as given
          (Holmes, Watson, Lucy, RACHE, John Ferrier, etc.).
        - Resolve pronouns ("he", "she", "they") to the named subject of the
          question whenever it is unambiguous.
        - Drop conversational filler ("could you tell me", "I want to know",
          "please", trailing politeness, question marks).
        - Phrase the output as a declarative search query, not a question.
        - One short sentence. No quotes. No preamble. No explanation.

      Below is an example:

      USER QUESTION:
      "Could you tell me what the word was that Holmes saw written on the wall in blood?"

      REWRITTEN QUERY:
      The word written in blood that Sherlock Holmes found on the wall at the crime scene

      Now rewrite the following question. Output only the rewritten query.

      USER QUESTION:
      {user_input}

      REWRITTEN QUERY:
    """

    # Generate the rewritten query at low temperature so the rewrite is stable
    rewritten_query = generate_completion(
        user_prompt=rewrite_prompt,
        model_name=generation_model,
        temperature=0.0,
    )

    return rewritten_query.strip()

In [19]:
# The following function uses rewrite_query as a lightweight pre-retrieval step
def do_query_rewrite_rag(
    user_input: str, top_k: int, verbose: bool = False
) -> list[FetchedChunk]:
    """Runs the retrieval pipeline using a single LLM-rewritten query

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    top_k : int
        The top_k closest chunks will be retrieved
    Returns
    -------
    context: [FetchedChunk]
        The fetched chunks
    """

    # Rewrite the user's question into a search-friendly query
    rewritten_query = rewrite_query(user_input=user_input)

    # Embed the rewritten query
    rewritten_query_embedding = do_embed(rewritten_query, embedding_model)

    # "R" like "Retrieval": Get the top-k semantically closest chunks to the rewritten query
    fetched_chunks = do_top_k_fetching(
        user_input_embedding=rewritten_query_embedding, top_k=top_k, chromadb_client=chromadb_client
    )

    if verbose:
        print(f"Original:  {user_input}")
        print(f"Rewritten: {rewritten_query}")
        print(f"Fetched chunks: {fetched_chunks}")
    return fetched_chunks

In [20]:
# TODO: Test the do_query_rewrite_rag function with a user input
user_input = "Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?"
context = do_query_rewrite_rag(user_input=user_input, top_k=DEFAULT_K, verbose=True)

Original:  Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?
Rewritten: The number written on the ceiling that Lucy noticed during breakfast
Fetched chunks: [FetchedChunk(chunk='some message or remonstrance from Young as to his conduct, and he was not mistaken, though it came in an unlooked-for manner. Upon rising next morning he found, to his surprise, a small square of paper pinned on to the coverlet of his bed just over his chest. On it was printed, in bold straggling letters:—  “Twenty-nine days are given you for amendment, and then——”  The dash was more fear-inspiring than any threat could have been. How this warning came into his room puzzled John Ferrier sorely, for his servants slept in an outhouse, and the doors and windows had all been secured. He crumpled the paper up and said nothing to his daughter, but the incident struck a chill into his heart. The twenty-nine days were evidently the balance of the month which Young ha

In [21]:
# TODO: Evaluate the new system. How does it perform compared to the baseline and to HyDE? What might be the reasons for the differences?
evaluation_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=testdata_items,
    rag_function=do_query_rewrite_rag,
)

print("Query rewriting evaluation results:")
print(f"Mean Hit Rate: {evaluation_result.mean_hit_rate:.2f}")
print(f"Mean Reciprocal Rank: {evaluation_result.mean_reciprocal_rank:.2f}")
print(f"Mean Precision: {evaluation_result.mean_precision:.2f}")

Query rewriting evaluation results:
Mean Hit Rate: 0.90
Mean Reciprocal Rank: 0.55
Mean Precision: 0.11


### Exercise 03: Hypothetical Document Embeddings (HyDE)
HyDE is a technique for improving retrieval. To do this, hypothetical documents are generated based on the user's input that could potentially solve the user's task. These generated documents should closely resemble the documents or chunks in the knowledge base. The hypothetical, generated documents are then used for semantic retrieval.  
The underlying idea is that, although the generated documents may not be factually accurate, they are often semantically closer to relevant knowledge base entries than the user's original query.  
In practice, multiple hypothetical documents are often generated, embedded, and then averaged over each embedding dimension to create a single embedding used for retrieval. This averaging is intended to neutralize incorrect facts while enhancing semantic similarity to the knowledge base.  
**Your task is to implement this method. For this task, you do not have to create more than one hypothetical document, so you can skip the averaging.**  
Afterwards, reflect on the potential drawbacks of this approach and evaluate the performance of the resulting system.

In [22]:
HYDE_GENERATION_MODEL = generation_model

# TODO: Implement a function which creates hypothetical paragraphs of Sherlock Holmes books from a given user input
def generate_hypothetical_documents(user_input: str) -> str:
    """ Generates a hypothetical book summary for a given user input

    Parameters
    ----------
    user_input : str
        The user input to be used for the generation
    Returns
    -------
    response: str
        The generated hypothetical book summary
    """

    # TODO: Use your prompt engineering skills to create a prompt for
    # generating a hypothetical Sherlock-Holmes-style passage from a given
    # user question.
    #
    # The goal is to produce a fictional text chunk written in the voice and
    # atmosphere of the Sherlock Holmes stories. The generated passage must:
    #   - Sound as if it could plausibly appear in an authentic Holmes tale
    #   - Include Holmes, Watson, clues, deductions, or London atmosphere
    #   - Contain information that directly answers the user’s question
    #   - Avoid quoting or summarizing real Sherlock Holmes texts
    #   - Focus on creating a believable, stylistically accurate *new* scene
    # As an example, you can use the following user question and the resulting
    # hypothetical passage:
    #
    # USER QUESTION:
    # "How did Holmes realize the visitor was lying about where he had spent the night?"
    #
    # HYPOTHETICAL PASSAGE:
    # Holmes stood beside the hearth, his gaze fixed not upon our nervous caller,
    # but upon the faint smear upon the man’s right cuff. “It is elementary, Watson,”
    # he remarked as the door closed behind the visitor. “He claims to have spent the
    # night at his club, yet that ash upon his sleeve is from a riverboat’s furnace—
    # coarser, darker, unmistakably mixed with coal dust. No gentleman’s club in
    # London makes use of such fuel.” Holmes paced to the window, hands clasped
    # behind his back. “And note the faint dampness on his boots. The Thames fog
    # was thick before dawn. Our friend was not reclining in a leather armchair:
    # he was skulking along the waterfront, and quite recently at that.”
    #
    # Use this example as a guide for the tone, length, and structure of the
    # passages your prompt should produce.

    hyde_prompt = f"""
      You are HYDE, a fictional generator of Sherlock Holmes-style text passages.
      Your purpose is to create hypothetical Sherlock Holmes text chunks that
      plausibly could appear in the original stories by Sir Arthur Conan Doyle.

      When a user asks a question about events, clues, characters, or deductions
      that might appear in a Sherlock Holmes story, you will generate a short,
      coherent passage written in the style of the Holmes canon. This passage
      should *sound like it comes from an authentic Holmes adventure* and
      should *contain information that directly answers the user’s question*.

      You are not summarizing any actual story, and you are not quoting or
      reproducing copyrighted text. Instead, you are inventing a new,
      Sherlock-Holmes-like scene that feels authentic in tone and structure:
      - Victorian London atmosphere
      - Holmes’s analytical reasoning
      - Watson’s narrative voice
      - Concrete clues, observations, and deductions
      - A sense of mystery and investigation

      Below is an example:

      USER QUESTION:
      "How did Holmes deduce the true identity of the man following us through the fog?"

      HYPOTHETICAL PASSAGE:
      As we turned into Baker Street, I noticed Holmes cast a brief glance
      over his shoulder, his sharp eyes glinting beneath the brim of his hat.
      “The fellow has altered his gait,” he murmured to me. “A simple matter, Watson:
      only a man trying too hard to appear inconspicuous walks with such irregularity.”
      The fog thickened as our pursuer paused by a lamppost, but Holmes had
      already reached his conclusion. “He bears the ink-stain of a government clerk
      on his cuff, and his boots are newly soled—military issue. Our mysterious shadow
      is none other than Inspector Aldwych in disguise, no doubt dispatched to verify
      the authenticity of our client’s claims.”

      Now, based on the following question, write a fictional Holmes-style passage
      in the same style and length. Ensure the invented passage contains details
      that would allow the true answer to be inferred.

      USER QUESTION:
      {user_input}

      HYPOTHETICAL PASSAGE:

    """

    # Generate a hypothetical book summary using the user input as a prompt
    hypothetical_summary = generate_completion(
        user_prompt=hyde_prompt,
        model_name=HYDE_GENERATION_MODEL,
        temperature=1.0,
    )

    return hypothetical_summary

In [23]:
# The following function now uses the new generate_hypothetical_documents function
def do_hyde_rag(
    user_input: str, top_k: int, n_hyde: int = 2, verbose: bool = False
) -> list[FetchedChunk]:
    """Runs the retrieval pipeline with a given user input and returns the context using the hyde approach

    Parameters
    ----------
    user_input : str
        The user input to be used for the RAG pipeline
    top_k : int
        The top_k closest chunks will be retrieved
    n_hyde : int
        The amount of hypothetical documents to be created
    Returns
    -------
    context: [str]
        The fetched chunks
    """

    # TODO: Generate a hypothetical paragraph using the user input as a prompt
    hypothetical_paragraphs = [
        generate_hypothetical_documents(user_input=user_input) for _ in range(n_hyde)
    ]

    # Embed the hypothetical paragraph
    hypothetical_paragraph_embeddings = [
        do_embed(hypothetical_paragraph, embedding_model)
        for hypothetical_paragraph in hypothetical_paragraphs
    ]

    # Average the embeddings
    dimension_embeddings = len(hypothetical_paragraph_embeddings[0])
    average_hypothetical_paragraph_embeddings = [
        sum(emb[i] for emb in hypothetical_paragraph_embeddings) / n_hyde
        for i in range(dimension_embeddings)
    ]

    # "R" like "Retrieval": Get the top-k semantically closest chunks to the hypothetical paragraph
    fetched_chunks = do_top_k_fetching(
        user_input_embedding=average_hypothetical_paragraph_embeddings, top_k=top_k, chromadb_client=chromadb_client
    )

    if verbose:
        print(f"User input: {user_input}")
        print(
            f"Example of generated hypothetical paragraph: {hypothetical_paragraphs[0]}"
        )
        print(f"Fetched chunks: {fetched_chunks}")
    return fetched_chunks

In [24]:
# TODO: Test the do_hyde_rag function with a user input
user_input = "Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?"
context = do_hyde_rag(user_input=user_input, top_k=DEFAULT_K, verbose=True)

User input: Lucy noticed a number on the ceiling when taking breakfast. Which number was written into the ceiling?
Example of generated hypothetical paragraph: It was during breakfast that Lucy, with her usual keen eye for domestic peculiarities, pointed towards a faint mark high on the plaster ceiling above the mantelpiece. "Just there, Mr. Holmes," she
Fetched chunks: [FetchedChunk(chunk='looking as merry and jovial as ever.  “‘Ah!’ said he, ‘you must not think me rude if I passed you without a word, my dear young lady. I was preoccupied with business matters.’  “I assured him that I was not offended. ‘By-the-way,’ said I, ‘you seem to have quite a suite of spare rooms up there, and one of them has the shutters up.’  “He looked surprised, and, as it seemed to me, a little startled at my remark.  “‘Photography is one of my hobbies,’ said he. ‘I have made my dark room up there. But, dear me! what an observant young lady we have come upon. Who would have believed it? Who would have ever

In [25]:
# TODO: Evaluate the new system. How does it perform compared to the original RAG system? What might be the reasons for the differences?
evaluation_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=testdata_items,
    rag_function=do_hyde_rag,
    sleep_seconds=0    # If you run into rate limits, you can try to increase this
)

print("HYDE evaluation results:")
print(f"Mean Hit Rate: {evaluation_result.mean_hit_rate:.2f}")
print(f"Mean Reciprocal Rank: {evaluation_result.mean_reciprocal_rank:.2f}")
print(f"Mean Precision: {evaluation_result.mean_precision:.2f}")

HYDE evaluation results:
Mean Hit Rate: 0.50
Mean Reciprocal Rank: 0.33
Mean Precision: 0.07


### (Optional) Exercise 04: Reranking
In a RAG system, rerankers play a crucial role in improving the quality of retrieved information before it's passed to the language model. While the retriever fetches potentially relevant documents based on semantic similarity, it often returns noisy or loosely related results. Rerankers evaluate and reorder these documents using more advanced scoring—often based on cross-encoders or task-specific relevance signals—ensuring that the most relevant, contextually appropriate content is prioritized. This leads to more accurate, coherent, and trustworthy generated outputs.  
For this exercise you will load a lightweight Hugging Face cross-encoder locally (no external API keys needed) and use it to rerank the fetched Sherlock Holmes chunks before passing them to the generator.

> **Setup:** this exercise needs `torch` and `transformers`, which aren't installed by default. From the project root, run `uv sync --group reranker` before executing the cells below.

In [26]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Unfortunately we can not use ollama here, since ollama does not provide the "rerank" endpoint yet.
# Therefore we need to handle this more 'low-level'.

reranker_model = 'BAAI/bge-reranker-v2-m3'
tokenizer = AutoTokenizer.from_pretrained(reranker_model)
model = AutoModelForSequenceClassification.from_pretrained(reranker_model)

def rerank(user_input: str, fetched_chunks: list[FetchedChunk]) -> list[FetchedChunk]:
    """Reranks fetched chunks using a sentence-transformer cross-encoder."""

    # Prepare pair inputs for scoring
    pair_inputs = [(user_input, chunk.chunk) for chunk in fetched_chunks]

    # Calculate similarity scores
    with torch.no_grad():
        inputs = tokenizer(pair_inputs, padding=True, truncation=True, return_tensors='pt', max_length=512)
        scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
    scored_chunks = sorted(
        zip(scores, fetched_chunks),
        key=lambda item: item[0],
        reverse=True
    )
    return [chunk for _, chunk in scored_chunks]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

In [ ]:
# TODO: Update the do_reranked_rag function to use the reranking function
def do_reranked_rag(user_input: str, top_k: int, verbose: bool = True) -> list[FetchedChunk]:
  """ Runs the RAG pipeline with a given user input and returns the response and the context

  Parameters
  ----------
  user_input : str
      The user input to be used for the RAG pipeline
  Returns
  -------
  response: str
      The generated response
  context: [str]
      The fetched chunks
  """

  # Embed the user input
  user_input_embedding = do_embed(user_input, embedding_model)

  # "R" like "Retrieval": Get the k semantically closest chunks to the user input from the knowledgebase
  fetched_chunks = do_top_k_fetching(user_input_embedding=user_input_embedding, top_k=top_k, chromadb_client=chromadb_client)

  # TODO: Rerank the fetched chunks based on the user input
  ranked_fetched_chunks = rerank(user_input=user_input, fetched_chunks=fetched_chunks)

  if verbose:
    print(f"User input: {user_input}")
    print("Fetched chunks:")
    for idx, c in enumerate(fetched_chunks):
      print(f"  [{idx}] {c.chunk[:80]}...")
    print("Ranked fetched chunks:")
    for idx, c in enumerate(ranked_fetched_chunks):
      print(f"  [{idx}] {c.chunk[:80]}...")
  return ranked_fetched_chunks

In [ ]:
# TODO: Evaluate the new system. How does it perform compared to the original RAG system?
evaluation_result = run_evaluation(
    top_k=DEFAULT_K,
    testdata_items=testdata_items,
    rag_function=do_reranked_rag
)
print(f"Mean Hit Rate: {evaluation_result.mean_hit_rate:.2f}")
print(f"Mean Reciprocal Rank: {evaluation_result.mean_reciprocal_rank:.2f}")
print(f"Mean Precision: {evaluation_result.mean_precision:.2f}")

User input: What year does Dr. Watson complete his medical degree?
Fetched chunks:
  [0] A STUDY IN SCARLET  By A. Conan Doyle     CONTENTS   A STUDY IN SCARLET.   PART ...
  [1] him.”  There was a movement and an exclamation from my right, and, peering throu...
  [2] his head sunk upon his chest and his hands clasped behind him. To me, who knew h...
  [3] he will be away all day, and that there would be nothing to disturb you. We have...
  [4] from the dead man’s lap, and throwing the noose round the reptile’s neck, he dre...
  [5] would be complete without some little sketch of this remarkable episode.  It was...
  [6] as she had entered, and we heard the wheels of her carriage rattle off down the ...
  [7] Holmes yet,” he said; “perhaps you would not care for him as a constant companio...
  [8] bell, Watson, and as the clock makes it a few minutes after four, I have no doub...
  [9] Isa Whitney, brother of the late Elias Whitney, D.D., Principal of the Theologic...
Ranked fetched ch